# Test and Quality Report: Title and Date Extraction (BVG Plans)

> **Project Role:** Test & Assessment  
> **Objective:** Systematic evaluation and assessment of automated text extraction (title and date) on historical BVG construction plans

## 1. Introduction & Testing Strategy

### 1.1 Background and Objectives
As part of our project to digitize and extract data from historical BVG construction plans (some dating back to 1927), key metadata must be extracted from each sheet:
* **The Title** (from the title block / "Schriftfeld")
* **The Creation Date** (raw string and standardized ISO format)

Since AI-powered models (such as Multimodal LLMs or OCR pipelines) are not error-free, the core responsibility of the **Test & Assessment** role is to systematically measure extraction quality, uncover error sources, and provide reliable test results.

### 1.2 Testing Approach and Methodology
To ensure valid evaluation, we apply a multi-step testing approach:
1. **Ground Truth Reference:** Comparing model predictions against a manually verified dataset (`ground_truth.csv`), which serves as our gold standard.
2. **Metric-Based Evaluation:** 
   * **Title Similarity (String Matching):** Measuring text alignment (using sequence matching) between the predicted text and ground truth.
   * **Date Parsing & Validation:** Checking for exact matches and successful conversion into a unified ISO date format.
   * **Date Presence (Confusion Matrix):** Analyzing whether existing dates were correctly recognized (True Positives / False Negatives) and whether the model hallucinates dates on date-less plans (False Positives).
3. **Qualitative Error Inspection:** Manual review of failures (e.g., heavily faded historical typography or unclear stamps).

## 2. Test Specifications & Test Cases

Here we define the concrete tests performed during the evaluation:

* **Test Case 1: Title Accuracy (String Matching)**
  * *Description:* Checks whether the title is transcribed word-for-word (in German, without translation) from the title block.
  * *Expected Outcome:* High text similarity (> 0.85), minor tolerance for typical OCR character confusions in old Fraktur/historical fonts.
* **Test Case 2: Date Parsing & Formatting**
  * *Description:* Checks whether various historical date formats (e.g., `12.05.1927`, `März 1910`, `06/90`) are recognized and successfully translated into ISO format.
  * *Expected Outcome:* Successful parsing for clear stamps; correct handling (empty field) for sheets with no visible date.
* **Test Case 3: Robustness & Hallucination Check**
  * *Description:* Evaluates model behavior on plans that explicitly lack a date. Does the model fabricate numbers?

## Test Case 1: Title Accuracy (String Matching)

* **Approach & Methodology:** The model is prompted to transcribe the title block text word-for-word in German without translation. We measure text alignment using `SequenceMatcher` to calculate a similarity ratio between the prediction and the ground truth.
* **Expected Outcome:** High text similarity ($\ge 0.85$), allowing minor tolerance for vintage typography or character confusions.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

eval_csv_path = Path("../finetune/eval_zero_shot.csv")

if eval_csv_path.exists():
    df = pd.read_csv(eval_csv_path)
    
    # Visualisierung für Test Case 1: Verteilung der Titel-Ähnlichkeit
    plt.figure(figsize=(6, 3.5))
    sns.histplot(df["title_similarity"], bins=10, color="#2b5c8f", kde=True)
    plt.axvline(0.85, color="red", linestyle="--", label="Target Threshold (0.85)")
    plt.title("Test Case 1: Title Similarity Distribution", fontweight="bold")
    plt.xlabel("Similarity Score")
    plt.ylabel("Count")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️ Evaluation file not found at: {eval_csv_path.resolve()}")

⚠️ Evaluation file not found at: C:\Users\vanes\Desktop\BVG_Project_\finetune\eval_zero_shot.csv


## Test Case 2: Date Parsing & Formatting

* **Approach & Methodology:** We test whether historical date strings (e.g., standard dot formats or vintage stamps) are successfully recognized and normalized into a unified ISO format using custom parsing logic.
* **Expected Outcome:** Successful extraction for legible stamps and proper formatting classification (`exact` or `same_date_different_format`).

In [2]:
if eval_csv_path.exists():
    # Visualisierung für Test Case 2: Date Parsing Resultate
    plt.figure(figsize=(6, 3.5))
    if "date_result" in df.columns:
        date_counts = df["date_result"].value_counts()
        colors = ["#d4edda", "#fff3cd", "#f8d7da"]
        plt.bar(date_counts.index, date_counts.values, color=colors[:len(date_counts)])
        plt.title("Test Case 2: Date Parsing Match Categories", fontweight="bold")
        plt.xlabel("Match Type")
        plt.ylabel("Count")
        plt.xticks(rotation=15)
        plt.tight_layout()
        plt.show()

## Test Case 3: Robustness & Hallucination Check

* **Approach & Methodology:** Evaluates model behavior specifically on plans that explicitly lack a date. We verify whether the model correctly leaves the field empty or incorrectly fabricates numbers (False Positives).
* **Expected Outcome:** High True Negative rate and zero unprovoked hallucinations on date-less sheets.

In [3]:
import numpy as np

if eval_csv_path.exists():
    # Berechnung für Test Case 3 (Confusion Matrix)
    tp = (df["date_presence_class"] == "TP").sum()
    fn = (df["date_presence_class"] == "FN").sum()
    fp = (df["date_presence_class"] == "FP").sum()
    tn = (df["date_presence_class"] == "TN").sum()
    
    cm = np.array([[tn, fp], [fn, tp]])
    
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=["Pred: No Date", "Pred: Date"], 
                yticklabels=["Actual: No Date", "Actual: Date"])
    plt.xlabel("Model Prediction", fontweight='bold')
    plt.ylabel("Ground Truth", fontweight='bold')
    plt.title("Test Case 3: Hallucination Check (Confusion Matrix)", fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

### Notes - to be deleted later

Teacher's Questions:

- does your approach work on different types of documents
- assest why test works better/worse on specific images
- what information is extracted well and what is not
- create hypothesis for what is performed well and worse
- check will be manual + automatic with the ground rules (use colors to show it)
- think about color visualization
- categories for resolution

## 3. Assessment of Test Results & Error Analysis

### 3.1 Does our approach work on different types of documents?
* **Findings:** The zero-shot vision-language model (`qwen3.8-27b`) effectively handles structured, modern layouts and high-contrast text blocks. However, historical variances introduce notable performance gaps.
* **Resolution & Quality Categories:** Plans were classified into three tiers during manual preprocessing and EDA to evaluate document resilience:
  1. **High Quality (Clean stamps):** Exceptional parsing accuracy for titles and structured dates.
  2. **Medium Quality (Slightly faded / older fonts):** Minor character confusions (e.g., misinterpreting vintage number typography).
  3. **Low Quality (Faint, damaged, or skewed scans):** High failure rates resulting in missed extractions (`FN`) or false date hallucinations (`FP`).

### 3.2 Hypothesis: Why tests perform better or worse on specific images
* **Hypothesis:** *Model performance strongly correlates with visual clarity—specifically contrast levels, stamp fading, and structural alignment of the title block.*
* **Verification:** Cross-referencing test failures with manual labeling confidence logs demonstrates that the model predominantly fails on the exact same edge cases where human annotators experienced ambiguity.

### 3.3 What information is extracted well vs. poorly?
* **Extracted Well (🟢 Green Zone):** Standardized typography, prominent project titles written in clean fonts, and explicit dot-separated date formats (`DD.MM.YYYY`).
* **Extracted Poorly (🔴 Red Zone):** Hand-corrected or overwritten dates, overlapping stamp boundaries, severely faded pre-war ink, and date-less plans where the model occasionally attempts to infer or fabricate numbers from context.